# Steady-vault selection explanation

This is the final human-readable decision sheet from `steady-vault-plan-01.md`.
It loads saved NB10--NB16 reports and ledgers, keeps vault address as the
primary identity, and writes the explanation sheet from those artefacts. It
does not reselect vaults, fit a model, or manufacture a clean holdout.

The sheet distinguishes point-in-time eligibility, requested weights and
accepted allocations. Young and sparse evidence remains visible through an
explicit cap; missing evidence is not treated as zero risk.


In [1]:
from pathlib import Path
import json
import pandas as pd

PROJECT_DIR = Path.cwd() / "scratchpad/hyperliquid-ic" if (Path.cwd() / "scratchpad/hyperliquid-ic").exists() else Path.cwd()
OUT = PROJECT_DIR / "_artifacts-selection-explanation"
OUT.mkdir(exist_ok=True)
ARTIFACT_DIRS = {
    "NB10": PROJECT_DIR / "_artifacts-steady-vault-data",
    "NB11": PROJECT_DIR / "_artifacts-profitability-repeatability",
    "NB12": PROJECT_DIR / "_artifacts-downside-stress",
    "NB13": PROJECT_DIR / "_artifacts-steady-vault-sizing",
    "NB14": PROJECT_DIR / "_artifacts-young-sparse",
    "NB15": PROJECT_DIR / "_artifacts-independent-groups",
    "NB16": PROJECT_DIR / "_artifacts-final-validation",
}

def load_report(name):
    path = ARTIFACT_DIRS[name] / f"{name.lower()}-report.json"
    if not path.exists():
        raise FileNotFoundError(path)
    return json.loads(path.read_text())

def result_count(report):
    value = report.get("metric_rows", report.get("backtest_metrics", report.get("metrics", [])))
    return int(value) if isinstance(value, (int, float)) else len(value)

REPORTS = {name: load_report(name) for name in ARTIFACT_DIRS}
coverage = pd.DataFrame([
    {"notebook": name, "report": str(ARTIFACT_DIRS[name] / f"{name.lower()}-report.json"),
     "metric_or_result_rows": result_count(report)}
    for name, report in REPORTS.items()
])
coverage.to_csv(OUT / "report-coverage.csv", index=False)
display(coverage)


,notebook,report,metric_or_result_rows
0,NB10,/Users/moo/code/getting-started/scratchpad/hyp...,16
1,NB11,/Users/moo/code/getting-started/scratchpad/hyp...,16
2,NB12,/Users/moo/code/getting-started/scratchpad/hyp...,32
3,NB13,/Users/moo/code/getting-started/scratchpad/hyp...,32
4,NB14,/Users/moo/code/getting-started/scratchpad/hyp...,12
5,NB15,/Users/moo/code/getting-started/scratchpad/hyp...,28
6,NB16,/Users/moo/code/getting-started/scratchpad/hyp...,18


## Frozen policy map

Requested weight is before TVL, individual, evidence or group caps. Accepted
weight is the post-cap amount in the saved target ledger.


In [2]:
POLICIES = pd.DataFrame([
    {"arm": "P0", "parent": "none", "rule": "positive observed growth", "purpose": "profitability screen"},
    {"arm": "P1", "parent": "P0", "rule": "growth >=15% annualised and positive first and second halves", "purpose": "profitability and repeatability"},
    {"arm": "P2", "parent": "P1", "rule": ">=70% positive observed weeks; largest event <=50%; best event removed remains positive", "purpose": "event repeatability"},
    {"arm": "D0", "parent": "P1", "rule": "P1 parity control", "purpose": "downside control"},
    {"arm": "D1", "parent": "P1", "rule": "max drawdown >= -3% on longest available 60/30/14-day window", "purpose": "downside admission"},
    {"arm": "D2", "parent": "D1", "rule": "annualised volatility <=8% on longest available 60/30/14-day window", "purpose": "low-volatility sensitivity"},
    {"arm": "D3", "parent": "D1", "rule": "halve requested weight after causal 180-day drawdown below -8%; released capital stays cash", "purpose": "stress response"},
    {"arm": "S0", "parent": "D1", "rule": "equal requested weights and 20% individual cap", "purpose": "sizing control"},
    {"arm": "S1", "parent": "D1", "rule": "inverse downside sizing with 5% floor and 20% cap", "purpose": "downside-aware sizing"},
    {"arm": "S2", "parent": "D1", "rule": "capped growth/downside sizing", "purpose": "sizing sensitivity"},
    {"arm": "E0", "parent": "P1 + D1", "rule": "20% mature cap; provisional young/sparse cap and 20% aggregate sleeve", "purpose": "young and sparse evidence"},
    {"arm": "E1", "parent": "P1 + D1", "rule": "2% individual provisional cap", "purpose": "strict evidence sensitivity"},
    {"arm": "E2", "parent": "P1 + D1", "rule": "graduated provisional cap from 2% to 10%", "purpose": "evidence sensitivity"},
    {"arm": "G0", "parent": "P1 + D1 + E0", "rule": "individual cap only", "purpose": "grouping control"},
    {"arm": "G1", "parent": "P1 + D1 + E0", "rule": "verified manager group cap 25%", "purpose": "manager concentration"},
    {"arm": "G2", "parent": "P1 + D1 + E0", "rule": "G1 plus complete-link weekly correlation >=0.75 with >=8 paired weeks", "purpose": "correlated concentration"},
])
POLICIES.to_csv(OUT / "policy-map.csv", index=False)
display(POLICIES)


,arm,parent,rule,purpose
0,P0,none,positive observed growth,profitability screen
1,P1,P0,growth >=15% annualised and positive first and...,profitability and repeatability
2,P2,P1,>=70% positive observed weeks; largest event <...,event repeatability
3,D0,P1,P1 parity control,downside control
4,D1,P1,max drawdown >= -3% on longest available 60/30...,downside admission
5,D2,D1,annualised volatility <=8% on longest availabl...,low-volatility sensitivity
6,D3,D1,halve requested weight after causal 180-day dr...,stress response
7,S0,D1,equal requested weights and 20% individual cap,sizing control
8,S1,D1,inverse downside sizing with 5% floor and 20% cap,downside-aware sizing
9,S2,D1,capped growth/downside sizing,sizing sensitivity


## Manually reviewed examples and accepted allocations

The reason field is a diagnostic classification based on the manual curve
review. It is not a label used by the selection engine. E0 allocations are
aggregated from the saved A0b target ledger by address and date.


In [3]:
identity = pd.read_csv(ARTIFACT_DIRS["NB10"] / "manual-example-identity.csv")
metrics = pd.read_csv(ARTIFACT_DIRS["NB10"] / "manual-example-metrics.csv")
metrics = metrics.loc[metrics["period"].eq("full")].copy()
targets = pd.read_parquet(ARTIFACT_DIRS["NB14"] / "targets-a0b_allowlist-E0-full.parquet")
targets["date"] = pd.to_datetime(targets["date"])

GROUPS = {
    "StratWise": "unknown / independent until verified",
    "PF1": "PF1", "Passivbot Canon": "Passivbot Canon",
    "FuturAI Medium": "FuturAI family (manual verified mapping)",
    "FuturAI Low": "FuturAI family (manual verified mapping)",
    "Citadel": "Citadel", "Satori Quantum": "Satori Quantum",
    "HYPErQuantum4": "unknown / independent until verified",
}

def reason(row):
    if row["label"] == "StratWise":
        return "provisional young reference; short history retained with evidence cap"
    if row["label"] in {"Citadel", "Satori Quantum"}:
        return "manual false-positive counterexample: jump dominated"
    if row["label"] == "HYPErQuantum4":
        return "manual false-negative candidate; excluded from retrospective A0b universe"
    return "manual steady candidate; subject to point-in-time P1/D1/E0 gates"

allocation = targets.groupby("address", as_index=False).agg(
    allocation_observations=("date", "size"), first_allocation_date=("date", "min"),
    last_allocation_date=("date", "max"), mean_requested_weight=("requested_weight", "mean"),
    max_requested_weight=("requested_weight", "max"), mean_accepted_weight=("accepted_weight", "mean"),
    max_accepted_weight=("accepted_weight", "max"), accepted_weight_days=("accepted_weight", lambda s: int((s > 0).sum())),
    provisional_days=("provisional", "sum"),
)
examples = identity.merge(metrics, on=["label", "address"], how="left")
examples = examples.merge(allocation, on="address", how="left")
examples["group"] = examples["label"].map(GROUPS).fillna("unknown")
examples["decision_reason"] = examples.apply(reason, axis=1)
examples["eligibility"] = examples["in_A0b"].map({True: "A0b allowlist", False: "outside retrospective A0b allowlist"})
examples["requested_weight"] = examples["mean_requested_weight"].fillna(0.0)
examples["accepted_allocation"] = examples["mean_accepted_weight"].fillna(0.0)
EXAMPLE_COLUMNS = [
    "label", "address", "eligibility", "decision_reason", "group", "annualised_growth",
    "annualised_one_day_vol", "annualised_downside", "max_drawdown", "best_event_removed_growth",
    "best_positive_event_share", "positive_observed_week_share", "observed_marks", "max_gap_days",
    "requested_weight", "accepted_allocation", "accepted_weight_days", "provisional_days",
    "first_allocation_date", "last_allocation_date",
]
EXAMPLES = examples[EXAMPLE_COLUMNS].sort_values("label")
EXAMPLES.to_csv(OUT / "manual-examples-decision-sheet.csv", index=False)
display(EXAMPLES)


,label,address,eligibility,decision_reason,group,annualised_growth,annualised_one_day_vol,annualised_downside,max_drawdown,best_event_removed_growth,best_positive_event_share,positive_observed_week_share,observed_marks,max_gap_days,requested_weight,accepted_allocation,accepted_weight_days,provisional_days,first_allocation_date,last_allocation_date
5,Citadel,0xda51323fe9800c8365646ad5c7ade0dd17fdc167,A0b allowlist,manual false-positive counterexample: jump dom...,Citadel,0.601914,0.365294,0.174798,-0.320577,0.307890,0.192191,0.276596,236,14.0,0.091539,0.091539,32.0,0.0,2026-02-26,2026-06-02
4,FuturAI Low,0xb65dd7c56afbf3b272ab5fc49be44b47dca18003,A0b allowlist,manual steady candidate; subject to point-in-t...,FuturAI family (manual verified mapping),0.101518,0.056102,0.034457,-0.033630,0.046953,0.217587,0.820513,199,1.0,0.075385,0.075385,1.0,0.0,2026-08-05,2026-08-05
3,FuturAI Medium,0x53375ca9a649f337d83d0834fec0db97640a2a06,A0b allowlist,manual steady candidate; subject to point-in-t...,FuturAI family (manual verified mapping),0.130383,0.103612,0.083490,-0.060248,0.044670,0.235962,0.875000,163,1.0,0.099714,0.050586,7.0,0.0,2026-05-31,2026-08-17
7,HYPErQuantum4,0xc16e0bf84ea8c2b85457010602cb6fa030bc1fd4,outside retrospective A0b allowlist,manual false-negative candidate; excluded from...,unknown / independent until verified,0.691694,0.161387,0.082831,-0.177584,0.539895,0.082867,0.815385,255,7.0,0.000000,0.000000,NaN,NaN,NaT,NaT
1,PF1,0xa1b6d8efbcb2fb750a84dbc05649fa4968034f04,A0b allowlist,manual steady candidate; subject to point-in-t...,PF1,0.389506,0.202592,0.117296,-0.135223,0.309542,0.069203,0.753846,254,7.0,0.089249,0.089249,44.0,0.0,2026-04-07,2026-09-12
2,Passivbot Canon,0x490af7d4a048a81db0f677517ed6373565b42349,A0b allowlist,manual steady candidate; subject to point-in-t...,Passivbot Canon,0.242664,0.227345,0.123524,-0.138457,0.138607,0.104187,0.661538,255,7.0,0.101856,0.050363,58.0,0.0,2026-05-17,2026-09-12
6,Satori Quantum,0xbbf7d7a9d0eaeab4115f022a6863450296112422,A0b allowlist,manual false-positive counterexample: jump dom...,Satori Quantum,0.369941,0.578857,0.397827,-0.426909,-0.161173,0.275003,0.492308,253,7.0,0.075911,0.075911,21.0,0.0,2026-04-17,2026-09-10
0,StratWise,0x0ff219ac20596b457558341bc410bc7a08a1394c,A0b allowlist,provisional young reference; short history ret...,unknown / independent until verified,0.283424,0.038988,0.018474,-0.006521,0.220936,0.141091,0.818182,59,1.0,0.095625,0.095625,10.0,0.0,2026-08-21,2026-09-12


## Nearest false cases and matched outcomes

These are nearest manually reviewed examples, rather than supervised labels.
The outcome table is a matched portfolio-level comparison from NB16 for the
same saved universe and period.


In [4]:
FALSE_CASES = EXAMPLES.loc[
    EXAMPLES["label"].isin(["Citadel", "Satori Quantum", "HYPErQuantum4", "StratWise", "PF1", "Passivbot Canon", "FuturAI Medium", "FuturAI Low"]),
    ["label", "address", "eligibility", "decision_reason", "annualised_growth", "max_drawdown", "best_event_removed_growth", "accepted_allocation"],
].copy()
FALSE_CASES["case_type"] = FALSE_CASES["label"].map({
    "Citadel": "false positive / jump counterexample", "Satori Quantum": "false positive / jump counterexample",
    "HYPErQuantum4": "false negative / universe exclusion",
}).fillna("steady reference or candidate")
FALSE_CASES.to_csv(OUT / "nearest-false-cases.csv", index=False)

outcomes = pd.read_csv(ARTIFACT_DIRS["NB16"] / "final-validation-metrics.csv")
outcomes["feasible_18pct_cagr"] = outcomes["cagr"] >= 0.18
MATCHED_OUTCOMES = outcomes[[
    "recipe", "recipe_label", "universe", "period", "cagr", "weekly_sharpe", "daily_volatility",
    "max_drawdown", "ulcer", "mean_cash_fraction", "mean_effective_positions", "mean_turnover",
    "feasible_18pct_cagr",
]].sort_values(["universe", "period", "weekly_sharpe"], ascending=[True, True, False])
MATCHED_OUTCOMES.to_csv(OUT / "matched-outcome-comparison.csv", index=False)
display(FALSE_CASES)
display(MATCHED_OUTCOMES)


,label,address,eligibility,decision_reason,annualised_growth,max_drawdown,best_event_removed_growth,accepted_allocation,case_type
5,Citadel,0xda51323fe9800c8365646ad5c7ade0dd17fdc167,A0b allowlist,manual false-positive counterexample: jump dom...,0.601914,-0.320577,0.307890,0.091539,false positive / jump counterexample
4,FuturAI Low,0xb65dd7c56afbf3b272ab5fc49be44b47dca18003,A0b allowlist,manual steady candidate; subject to point-in-t...,0.101518,-0.033630,0.046953,0.075385,steady reference or candidate
3,FuturAI Medium,0x53375ca9a649f337d83d0834fec0db97640a2a06,A0b allowlist,manual steady candidate; subject to point-in-t...,0.130383,-0.060248,0.044670,0.050586,steady reference or candidate
7,HYPErQuantum4,0xc16e0bf84ea8c2b85457010602cb6fa030bc1fd4,outside retrospective A0b allowlist,manual false-negative candidate; excluded from...,0.691694,-0.177584,0.539895,0.000000,false negative / universe exclusion
1,PF1,0xa1b6d8efbcb2fb750a84dbc05649fa4968034f04,A0b allowlist,manual steady candidate; subject to point-in-t...,0.389506,-0.135223,0.309542,0.089249,steady reference or candidate
2,Passivbot Canon,0x490af7d4a048a81db0f677517ed6373565b42349,A0b allowlist,manual steady candidate; subject to point-in-t...,0.242664,-0.138457,0.138607,0.050363,steady reference or candidate
6,Satori Quantum,0xbbf7d7a9d0eaeab4115f022a6863450296112422,A0b allowlist,manual false-positive counterexample: jump dom...,0.369941,-0.426909,-0.161173,0.075911,false positive / jump counterexample
0,StratWise,0x0ff219ac20596b457558341bc410bc7a08a1394c,A0b allowlist,provisional young reference; short history ret...,0.283424,-0.006521,0.220936,0.095625,steady reference or candidate


,recipe,recipe_label,universe,period,cagr,weekly_sharpe,daily_volatility,max_drawdown,ulcer,mean_cash_fraction,mean_effective_positions,mean_turnover,feasible_18pct_cagr
17,A0b_baseline,A0b independent replay,a0b_allowlist,full,0.158638,0.841489,0.177131,-0.085948,0.028737,0.033533,NaN,22248.396333,False
7,B_P2,P2 (without D1/E0),a0b_allowlist,full,-0.088324,-0.486168,0.167860,-0.158818,0.082581,0.429440,8.172190,24423.167542,False
11,C_S1,S1 inverse downside,a0b_allowlist,full,-0.078504,-0.709117,0.079842,-0.104861,0.063323,0.605702,4.563002,10209.968922,False
15,D_G2,G2 independent groups,a0b_allowlist,full,-0.071637,-0.721426,0.074678,-0.111885,0.062254,0.637113,4.625178,10125.075594,False
3,A_P1D1E0,P1+D1+E0,a0b_allowlist,full,-0.075502,-0.769188,0.075424,-0.111949,0.062420,0.616368,4.926409,10644.623905,False
16,A0b_baseline,A0b independent replay,a0b_allowlist,hyper_ai,0.508794,1.984167,0.175305,-0.073587,0.017767,0.029070,NaN,26432.312348,True
10,C_S1,S1 inverse downside,a0b_allowlist,hyper_ai,-0.154414,-1.053850,0.109564,-0.104496,0.067075,0.479588,5.982950,13459.158238,False
14,D_G2,G2 independent groups,a0b_allowlist,hyper_ai,-0.140390,-1.067959,0.102082,-0.102512,0.062207,0.527052,6.284720,13310.238589,False
2,A_P1D1E0,P1+D1+E0,a0b_allowlist,hyper_ai,-0.144630,-1.109412,0.103242,-0.103872,0.061863,0.495615,6.584294,14422.328386,False
6,B_P2,P2 (without D1/E0),a0b_allowlist,hyper_ai,-0.201222,-1.387241,0.157765,-0.132913,0.063296,0.450111,8.767511,26648.579933,False


## Accounting, clocks and caveats

* Selection is point-in-time and uses observations available before the decision timestamp.
* Weekly Sharpe is the primary comparison clock; daily metrics are secondary diagnostics.
* Fresh raw observations are forward-filled for valuation only; missing returns are not zero risk.
* Share-price NAV is the accounting input and no additional fees are added; internalised vault fees remain in NAV.
* A0b is a retrospective current allowlist and requires the full-panel sensitivity. The held-NAV ledger is a mark diagnostic, not a liquidation replay.
* Young or sparse vaults can be provisionally selected with lower caps; StratWise remains the explicit young reference.
* Verified manager and relatedness groups are used only where provenance is available; unknown names stay singleton groups with the declared provisional cap.
* Best-event removal is an attribution diagnostic, not a tradable counterfactual. Historical results do not establish prospective performance.


In [5]:
SUMMARY_PATH = PROJECT_DIR / "steady-vault-summary-01.md"
summary = [
    "# Steady-vault selection summary", "",
    "Generated by `17-research-steady-vault-selection-explanation.ipynb` from saved NB10--NB16 artefacts.", "",
    "## Decision sheet", "",
    "The example table reports address identity, manual eligibility/reason, full-period metrics, requested E0 weight and accepted E0 allocation. Accepted allocations are post-cap observations from the saved target ledger.", "",
    EXAMPLES.to_markdown(index=False), "",
    "## Candidate policy map", "", POLICIES.to_markdown(index=False), "",
    "## Nearest false cases", "", FALSE_CASES.to_markdown(index=False), "",
    "## Matched portfolio outcomes", "", MATCHED_OUTCOMES.to_markdown(index=False), "",
    "## Interpretation", "",
    "Eligibility, requested allocation and accepted allocation are separate. Young and sparse evidence remains visible without a long-history admission barrier. The final comparison is retrospective: A0b is survivor-conditioned, discovery dates are not a clean holdout, and the next validation step is a prospective shadow run.", "",
]
SUMMARY_PATH.write_text("\n".join(summary) + "\n")
report = {
    "notebook": "17-research-steady-vault-selection-explanation.ipynb",
    "source_reports": {name: str(ARTIFACT_DIRS[name] / f"{name.lower()}-report.json") for name in ARTIFACT_DIRS},
    "manual_example_rows": int(len(EXAMPLES)), "false_case_rows": int(len(FALSE_CASES)),
    "matched_outcome_rows": int(len(MATCHED_OUTCOMES)),
    "stratwise_address": str(EXAMPLES.loc[EXAMPLES["label"].eq("StratWise"), "address"].iloc[0]),
    "summary_path": str(SUMMARY_PATH),
    "artifacts": ["report-coverage.csv", "policy-map.csv", "manual-examples-decision-sheet.csv", "nearest-false-cases.csv", "matched-outcome-comparison.csv", "nb17-report.json"],
    "assumptions": {
        "selection_clock": "point-in-time decision timestamps", "primary_ranking_clock": "weekly Sharpe, daily secondary",
        "valuation": "fresh raw marks forward-filled for valuation only", "fees": "no additional fees; share-price NAV includes internalised fees",
        "universe": "A0b retrospective allowlist plus full-panel sensitivity", "young_sparse": "provisional evidence caps; no long-history admission barrier",
    },
}
(OUT / "nb17-report.json").write_text(json.dumps(report, indent=2) + "\n")
display(pd.DataFrame([report]))


,notebook,source_reports,manual_example_rows,false_case_rows,matched_outcome_rows,stratwise_address,summary_path,artifacts,assumptions
0,17-research-steady-vault-selection-explanation...,{'NB10': '/Users/moo/code/getting-started/scra...,8,8,18,0x0ff219ac20596b457558341bc410bc7a08a1394c,/Users/moo/code/getting-started/scratchpad/hyp...,"[report-coverage.csv, policy-map.csv, manual-e...",{'selection_clock': 'point-in-time decision ti...
